# McDonald-Dunn Research Forest: Forest Recovery from Active Timber Management 1985–2025
### Using USFS Landscape Change Monitoring System (LCMS) in Google Earth Engine

---

**Research Question:** How do actively managed compartments at the McDonald-Dunn Research Forest cycle through LCMS-detectable succession stages, and can 40 years of LCMS data capture the full harvest-to-recovery arc for a commercially managed Oregon Coast Range forest?

**Dataset:** [LCMS v2025-11](https://developers.google.com/earth-engine/datasets/catalog/projects_gtac-data-publish_assets_LCMS_Product_Version_2025-11) — USFS/GTAC annual land cover, land use, and change maps at 30 m, 1985–2025 (CONUS + SE Alaska).

LCMS produces three annual thematic products:
| Band | What it shows |
|------|--------------|
| `Land_Cover` | What is on the ground (Trees, Shrubs, Grass, Barren, Water, etc.) |
| `Land_Use` | How the land is used (Forest, Agriculture, Developed, Rangeland, etc.) |
| `Change` | What changed and how (Tree Removal, Successional Growth, Wildfire, Stable, etc.) |

**Why the McDonald-Dunn Research Forest?**  
The [McDonald-Dunn Research Forest](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests) is an 11,500-acre teaching and demonstration forest operated by Oregon State University's College of Forestry, located approximately 15 minutes north of the OSU Corvallis campus in the Oregon Coast Range foothills. Unlike LTER sites focused on ecological mechanics, McDonald-Dunn is a **financially self-sustaining, actively managed** forest — timber harvests are conducted throughout the LCMS period on a rotation cycle, making it an ideal site for studying how LCMS captures commercial harvest events and the subsequent recovery arc in a Coast Range Douglas-fir setting.

Key characteristics relevant to LCMS analysis:
- **Continuous active management**: Clearcut, shelterwood, seed-tree, and selection harvests documented throughout 1985–2025
- **Multiple rotation cohorts**: Stands span from early 1950s OSU plantations to harvests within the last decade, creating a rich chronosequence
- **Complex landscape mosaic**: The constant rotation creates a highly dynamic patchwork — in sharp contrast to LTER sites managed for long-term stability
- **Oregon Coast Range Douglas-fir type**: Lower elevation than the western Cascades, with faster early seral transitions due to warmer temperatures and abundant moisture
- **Layered disturbance history**: From Kalapuya burning, to late-1800s settler harvest, to WWII military use (Dunn Forest / Camp Adair), to modern OSU silvicultural research

LCMS covers 1985–2025, which means:
- Areas harvested **before 1985** appear at the *start* of the record already in mid-recovery (older OSU plantations established in the 1950s–1960s)
- Areas harvested **during 1985–2025** show their full post-harvest trajectory captured in LCMS
- Long-uncut old-growth remnant patches provide stable reference pixels throughout

**Prerequisites**
- A Google Earth Engine (GEE) account — [sign up here](https://earthengine.google.com/signup/)
- A GEE Cloud project ID — [create one here](https://console.cloud.google.com/projectcreate)
- Python ≥ 3.9 with `earthengine-api` and `geeViz` installed:
  ```bash
  pip install earthengine-api geeViz
  ```

> 💡 **Workshop note:** This notebook is designed to run sequentially top-to-bottom. All cells are self-documenting. Run `Kernel → Restart & Run All` for a clean start.

## 1 · Setup — Imports and Authentication

In [21]:
import os, pathlib, json, ee
from IPython.display import display, HTML

# ── Authentication ────────────────────────────────────────────────────────────
#ee.Authenticate()

EE_PROJECT = 'rcr-gee'
ee.Initialize(project=EE_PROJECT)

os.environ['GEEVIZ_EEAUTH_MODE'] = 'auto'

import geeViz.getImagesLib as gil
import geeViz.geeView
import geeViz.getSummaryAreasLib as sal
from geeViz.outputLib import charts as cl

Map = gil.Map
Map.port = 8080
Map.project = EE_PROJECT
Map.clearMap()

test = ee.Image(1).getInfo()
print(test, '\n Earth Engine initialized successfully.')


{'type': 'Image', 'bands': [{'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]} 
 Earth Engine initialized successfully.


## 2 · Study Area — McDonald-Dunn Research Forest

The [McDonald-Dunn Research Forest](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests) encompasses approximately 11,500 acres (4,650 ha) in the Oregon Coast Range foothills, roughly 10 miles north of the OSU Corvallis campus. The combined forest comprises two blocks with distinct histories:

- **McDonald Forest** (~5,000 acres, southern block): acquired beginning in 1926 with donations from Mary McDonald; first harvested by Euro-American settlers in the late 1800s; now carries multiple generations of OSU-managed Douglas-fir plantations
- **Dunn Forest** (~6,500 acres, northern block): former Camp Adair WWII military training ground, acquired by OSU in 1947; prior agricultural and military land use created a complex early land-cover mosaic; bullets can occasionally still be found in older trees

The forest's actively managed history creates a highly dynamic landscape well suited to LCMS analysis:

| Period | Management Activity | LCMS Visibility |
|--------|--------------------|----|
| Pre-1985 | Multiple rotation cycles since OSU acquisition (~1930s onward); early settler harvest; Dunn Forest recovery from WWII-era land use | Mixed-age plantations at varying recovery stages at LCMS start |
| 1985–1994 | Active commercial timber harvest on rotation across numerous compartments; diverse silvicultural treatments for teaching | Tree Removal and Successional Growth signals across multiple compartments per year |
| 1994–2010 | Continued active harvest under updated management plans — unlike adjacent USFS lands, **not** affected by the 1994 Northwest Forest Plan | Ongoing Tree Removal events; mid-rotation stands entering crown closure |
| 2010–2025 | Recent clearcut, shelterwood, and partial harvests; ongoing active management | Tree Removal near end of record; early-seral Shrub/Grass visible; Trees recovery rising |

Because the forest is continuously and actively managed, Tree Removal events should appear somewhere in the study area in nearly every year of the LCMS record — a pattern not visible at LTER or wilderness sites.

> 📍 **Polygon note:** The harvest unit boundaries used in this notebook are approximate. For analysis using verified boundaries, contact the OSU Research Forest Business Office at Peavy Arboretum or consult the [2025 Forest Management Plan](https://cf.forestry.oregonstate.edu/our-forests/2025-mcdonald-dunn-forest-plan).

In [ ]:
START_YEAR = 1984
END_YEAR   = 2024

# ── Export format flags ───────────────────────────────────────────────────────
EXPORT_HTML = False   # save interactive Plotly/geeViz charts as .html
EXPORT_CSV  = False   # save underlying DataFrames as .csv
EXPORT_DIR  = pathlib.Path(r'C:\Users\LILA\Documents\GitHub\lcms-outreach\exports')
# Set EXPORT_DIR = None to save alongside this notebook instead

study_area = ee.Geometry.BBox(-123.46, 44.60, -123.22, 44.82)

print("Study date range:", START_YEAR, "to", END_YEAR)
print('Study area type:', study_area.getInfo()['type'])
area_km2 = study_area.area(maxError=500).divide(1e6).getInfo()
print(f'Bounding box area : {area_km2:,.0f} km2  (~46.5 km2 combined forest; bbox includes surrounding context)')

# ── McDonald-Dunn local GeoJSON files ─────────────────────────────────────────
_data_dir = pathlib.Path('..') / 'data'   # relative to examples/

with open(_data_dir / 'macdunn_boundary.geojson') as _f:
    _boundary_gj = json.load(_f)
macdunn_boundary_fc = ee.FeatureCollection(_boundary_gj['features'])

with open(_data_dir / 'macdunn_age_class.geojson') as _f:
    _stands_gj = json.load(_f)
macdunn_stands_fc = ee.FeatureCollection(_stands_gj['features'])

print(f'Boundary features: {macdunn_boundary_fc.size().getInfo()}')
print(f'Stand features   : {macdunn_stands_fc.size().getInfo()}')

# Resolve export directory
import os as _os
_notebook_dir = _os.path.dirname(_os.path.abspath('macdunn_harvest_recovery.ipynb'))
_export_dir   = str(EXPORT_DIR) if EXPORT_DIR is not None else _notebook_dir
if EXPORT_DIR is not None:
    pathlib.Path(_export_dir).mkdir(parents=True, exist_ok=True)
print(f'Output directory : {_export_dir}')


Study date range: 1984 to 2024


## 3 · Load LCMS Data

LCMS v2025-11 covers 1985–2025 (41 years). Each image in the collection represents one calendar year. We filter to the McDonald-Dunn bounding box and inspect what's available.

For the Oregon Coast Range, the LCMS `Change` band classes most relevant to this analysis are:

| Change Class | Ecological Meaning |
|---|---|
| **Tree Removal** | Timber harvest or clearing — the primary disturbance of interest; expect this to appear in many years across different compartments in an actively managed forest |
| **Vegetation Successional Growth** | Forest recovery — canopy closure and stand development after harvest; the positive counterpart to Tree Removal in the rotation cycle |
| **Wildfire** | Fire disturbance — relatively rare in the wet Coast Range but possible during drought years |
| **Stable** | No detected change — mature stands and older plantations hold this class between harvests |

> 🔍 The cell below prints the full class-value mapping so you can see the exact numeric IDs used in your LCMS version. These values are read directly from the image metadata rather than hardcoded.

In [ ]:
LCMS_ASSET_FOR_PROPERTIES = 'USFS/GTAC/LCMS/v2024-10'
LCMS_ASSET = 'projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11'

lcms = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).filterBounds(study_area)

n_images = lcms.size().getInfo()
years    = lcms.aggregate_array('year').distinct().sort().getInfo()
bands    = lcms.first().bandNames().getInfo()

print(f'Images in collection : {n_images}')
print(f'Years                : {years[0]}-{years[-1]}')
print(f'Bands                : {bands}')

sample_img      = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).first()
change_names    = sample_img.get('Change_class_names').getInfo()
change_values   = sample_img.get('Change_class_values').getInfo()
change_palettes = sample_img.get('Change_class_palette').getInfo()

change_img = lcms.first().select('Change')
print('Change image properties:')
for key in change_img.propertyNames().getInfo():
    print(f'  {key}: {change_img.get(key).getInfo()}')


c:\Users\LILA\Documents\GitHub\lcms-outreach\.venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for USFS/GTAC/LCMS/v2024-10! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10

  warnings.warn(warning, category=DeprecationWarning)


Images in collection : 40
Years                : 1985-2024
Bands                : ['Change', 'Land_Cover', 'Land_Use', 'Change_Raw_Probability_Slow_Loss', 'Change_Raw_Probability_Fast_Loss', 'Change_Raw_Probability_Gain', 'Land_Cover_Raw_Probability_Trees', 'Land_Cover_Raw_Probability_Tall-Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Trees-Mix', 'Land_Cover_Raw_Probability_Barren-and-Trees-Mix', 'Land_Cover_Raw_Probability_Tall-Shrubs', 'Land_Cover_Raw_Probability_Shrubs', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Barren-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb', 'Land_Cover_Raw_Probability_Barren-and-Grass-Forb-Herb-Mix', 'Land_Cover_Raw_Probability_Barren-or-Impervious', 'Land_Cover_Raw_Probability_Snow-or-Ice', 'Land_Cover_Raw_Probability_Water', 'Land_Use_Raw_Probability_Agriculture', 'Land_Use_Raw_Probability_Developed', 'Land_Use_Raw_Prob

## 4 · Interactive Map — Current Forest State and Harvest History

The map below shows three layers for the McDonald-Dunn Research Forest:

1. **Land Cover 2025** — what is on the ground today; the active rotation management creates a striking patchwork of Trees, Shrubs, and early-seral vegetation
2. **Most Common Change Agent (1985–2025)** — the modal LCMS Change class for each pixel across the full record
3. **Ever Harvested 1985–2025** — pixels flagged as "Tree Removal" in *any* year of the 41-year LCMS record

The **Ever Harvested** layer will show a much larger and more spatially distributed pattern than at a comparable LTER or wilderness site — the expected signature of an actively managed teaching forest where harvests occur across multiple compartments every decade.

> 🗺️ Toggle layers on/off with the layer panel on the right. Click any pixel to query its class value. Draw a polygon and click "Chart Selected Area" to analyze custom sub-regions.

In [ ]:
Map.clearMap()

lcms_most_recent = lcms.filter(ee.Filter.eq('year', END_YEAR))

Map.addLayer(
    lcms_most_recent.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}',
    True,
)

# Most recent change agent
lcms_most_recent_change = lcms_most_recent.select('Change').first().clip(study_area)
Map.addLayer(
    lcms_most_recent_change,
    {'autoViz': True, 'canAreaChart': True},
    f'Most Recent Change Agent ({END_YEAR})',
    True,
)

# Most common Change agent — .mode() drops metadata; copyProperties() restores it
lcms_most_common_change = (
    ee.Image(lcms.select('Change').mode())
    .clip(study_area)
    .copyProperties(sample_img)
)
Map.addLayer(
    lcms_most_common_change,
    {'autoViz': True, 'canAreaChart': True},
    'Most Common Change Agent (1985-2025)',
    False,
)

# Most severe Change agent — reducer appends '_min'; rename back for autoViz
lcms_most_severe_change = (
    ee.Image(lcms.select('Change').reduce(ee.Reducer.min()))
    .rename('Change')
    .clip(study_area)
    .copyProperties(sample_img)
)
Map.addLayer(
    lcms_most_severe_change,
    {'autoViz': True, 'canAreaChart': True},
    'Most Severe Change Agent (1985-2025)',
    False,
)

name_to_val = dict(zip(change_names, change_values))
TREE_REMOVAL_VAL = name_to_val.get('Tree Removal', name_to_val.get('Non-Fire Mechanical', 5))
print(f'Tree Removal class value: {TREE_REMOVAL_VAL}')

# Ever harvested mask
ever_harvested = (
    lcms.select('Change')
    .map(lambda img: img.eq(TREE_REMOVAL_VAL))
    .max().selfMask().rename('ever_harvested')
)
ever_harvested = ever_harvested.set({
    'ever_harvested_class_values':  [1],
    'ever_harvested_class_names':   ['Tree Removal detected (any year 1985-2025)'],
    'ever_harvested_class_palette': ['c47b1e'],
})
Map.addLayer(ever_harvested, {'autoViz': True}, 'Ever Harvested 1985-2025', True)

# Most recent harvest year per pixel
harvest_year = (
    lcms.select('Change')
    .map(lambda img: ee.Image.constant(ee.Number(img.get('year')))
                       .toFloat()
                       .updateMask(img.eq(TREE_REMOVAL_VAL))
                       .rename('harvest_year'))
    .max()
)
Map.addLayer(
    harvest_year,
    {'min': START_YEAR, 'max': END_YEAR,
     'palette': ['ffffb2', 'fecc5c', 'fd8d3c', 'f03b20', 'bd0026']},
    'Most Recent Harvest Year',
    False,
)

# Stand age raster (young = light blue, old = dark blue)
age_raster = (
    macdunn_stands_fc
    .reduceToImage(properties=['Age'], reducer=ee.Reducer.first())
    .rename('Age')
)
Map.addLayer(
    age_raster,
    {'min': 9, 'max': 340,
     'palette': ['dce9f5', '9ecae1', '4292c6', '2171b5', '08519c', '08306b']},
    'Stand Age (years)',
    False,
)

Map.addLayer(
    macdunn_stands_fc,
    {'layerType': 'geeVectorImage', 'strokeColor': '00000066', 'strokeWidth': 0.5,
     'fillColor': '00000000'},
    'McDonald-Dunn Stands',
    True,
)
Map.addLayer(
    macdunn_boundary_fc,
    {'layerType': 'geeVectorImage', 'strokeColor': 'cccccc', 'strokeWidth': 1,
     'fillColor': '00000000'},
    'McDonald-Dunn Boundary',
    True,
)
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffff00', 'strokeWidth': 2.5,
     'fillColor': '00000000'},
    'McDonald-Dunn BBox',
    False,
)

Map.setCenter(-123.34, 44.70, 11)
Map.view()


Adding layer: Land Cover 2024
Adding layer: Most Recent Change Agent (2024)
Adding layer: Most Common Change Agent (1985-2025)
Adding layer: Most Severe Change Agent (1985-2025)
Tree Removal class value: 9
Adding layer: Ever Harvested 1985-2025
Adding layer: Most Recent Harvest Year
Adding layer: Stand Age (years)
Adding layer: McDonald-Dunn Stands
Adding layer: McDonald-Dunn Boundary
Adding layer: McDonald-Dunn BBox
Starting webmap


[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8890/ee-api (tenant_header=X-geeViz-Creds)


geeViz server at http://localhost:8080/geeView/
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784663611854


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Build a DataFrame from the local stand attributes
# Focus on stands within the early-recovery window (Age 9-60 yrs)
stands_df = pd.DataFrame([
    f['properties'] for f in _stands_gj['features']
    if f.get('properties') and f['properties'].get('Age_Class')
])
stands_df['Area_ha'] = stands_df['Shape__Area'] / 10000

# 4 age classes covering early-to-stabilizing recovery dynamics (9-60 yrs)
# Colors match the recovery stage palette used in Section 11
_AGE_CLASS_LABELS = ['9-20', '21-35', '36-50', '51-60']
_AGE_CLASS_COLORS = ['#f7dc6f', '#e67e22', '#27ae60', '#1a5276']
_AGE_CLASS_NAMES  = [
    '9–20 yrs (pioneer → shrub)',
    '21–35 yrs (canopy closure)',
    '36–50 yrs (young forest)',
    '51–60 yrs (maturing stand)',
]

summary = (
    stands_df.groupby('Age_Class', observed=True)
    .agg(Stand_Count=('FID', 'count'), Total_Area_ha=('Area_ha', 'sum'))
    .reindex(_AGE_CLASS_LABELS)
    .reset_index()
)

fig_hist = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Number of Stands by Age Class', 'Total Area (ha) by Age Class'),
    horizontal_spacing=0.12,
)

for i, row in summary.iterrows():
    color = _AGE_CLASS_COLORS[i]
    label = _AGE_CLASS_NAMES[i]
    fig_hist.add_trace(go.Bar(
        x=[label], y=[row['Stand_Count']],
        marker_color=color, showlegend=False, name=label,
    ), row=1, col=1)
    fig_hist.add_trace(go.Bar(
        x=[label], y=[round(row['Total_Area_ha'], 1)],
        marker_color=color, showlegend=False, name=label,
    ), row=1, col=2)

fig_hist.update_layout(
    title='McDonald-Dunn Research Forest — Early-Recovery Stand Age Distribution (9–60 yrs)',
    xaxis=dict(title='Age Class'),  xaxis2=dict(title='Age Class'),
    yaxis=dict(title='Number of stands'), yaxis2=dict(title='Total area (ha)'),
    width=1000, height=420,
    template='plotly_white',
)
fig_hist.show()

print(summary.to_markdown(index=False))


| Age_Class   |   Stand_Count |   Total_Area_ha |
|:------------|--------------:|----------------:|
| 9-20        |            99 |        2464.09  |
| > 20-40     |            84 |        1401.18  |
| > 40-80     |           230 |        3775.63  |
| > 80-120    |            84 |        1625.11  |
| > 120-200   |            57 |        1144.16  |
| > 200-500   |            15 |         420.223 |


In [ ]:
# ── Granular age distribution — individual stand ages on x-axis (9-60 yr window)
forest_df = stands_df[~stands_df['STANDID'].str.startswith('AGLAND')].copy()

_CLASS_COLOR_MAP = dict(zip(_AGE_CLASS_LABELS, _AGE_CLASS_COLORS))
forest_df['bar_color'] = forest_df['Age_Class'].map(_CLASS_COLOR_MAP).fillna('#dddddd')
forest_df = forest_df.sort_values('Age')

fig_age = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Stand Count by Age (each bar = one stand)', 'Stand Area (ha) by Age'),
    vertical_spacing=0.12,
    shared_xaxes=True,
)

fig_age.add_trace(go.Bar(
    x=forest_df['Age'],
    y=[1] * len(forest_df),
    marker_color=forest_df['bar_color'],
    showlegend=False,
    hovertemplate='Age: %{x} yrs<br>Stand: %{customdata[0]}<br>Age Class: %{customdata[1]}',
    customdata=forest_df[['STANDID', 'Age_Class']].values,
    name='Stand count',
), row=1, col=1)

fig_age.add_trace(go.Bar(
    x=forest_df['Age'],
    y=forest_df['Area_ha'].round(1),
    marker_color=forest_df['bar_color'],
    showlegend=False,
    hovertemplate='Age: %{x} yrs<br>Area: %{y} ha<br>Stand: %{customdata[0]}',
    customdata=forest_df[['STANDID']].values,
    name='Stand area (ha)',
), row=2, col=1)

for label, color, name in zip(_AGE_CLASS_LABELS, _AGE_CLASS_COLORS, _AGE_CLASS_NAMES):
    fig_age.add_trace(go.Bar(
        x=[None], y=[None],
        marker_color=color,
        name=name,
        showlegend=True,
    ), row=1, col=1)

# Mark the LCMS observation window boundary at age 40 (stands ~40 yrs old in 1985
# were established ~1945 — their entire rotation is captured in the LCMS record)
fig_age.add_vline(x=40, line_dash='dash', line_color='gray', opacity=0.5,
                  annotation_text='≈ full LCMS window', annotation_position='top right',
                  row='all', col=1)

fig_age.update_layout(
    title='McDonald-Dunn Research Forest — Stand Age Distribution (early-recovery focus, 9–60 yrs)',
    xaxis2=dict(title='Stand age (years)', range=[8, 62], dtick=5),
    yaxis=dict(title='Number of stands'),
    yaxis2=dict(title='Stand area (ha)'),
    barmode='stack',
    legend=dict(
        title='Age Class', orientation='h',
        yanchor='top', y=-0.14, xanchor='left', x=0,
    ),
    width=1000, height=600,
    template='plotly_white',
)
fig_age.show()


## 5 · Age-Class Representative Stands

We focus on four age classes spanning the **early-to-stabilizing recovery window (9–60 years)** — the range where LCMS captures the most dynamic post-harvest trajectories. Each class gets one representative stand (largest non-agricultural stand in the bin).

| Stand | Age Class | Stand Age | Area | Recovery Stage |
|-------|-----------|-----------|------|----------------|
| **Stand 020408** | 9–20 yrs | 15 yrs | 81.5 ha | Pioneer → early shrub; Tree Removal near end of record; Grass/Herb dominant |
| **Stand 010207** | 21–35 yrs | 21 yrs | 75.2 ha | Canopy closure beginning; Trees % rising from near-zero; Successional Growth dominant |
| **Stand 010105** | 36–50 yrs | 49 yrs | 53.4 ha | Young forest establishing; Trees dominant but harvest may appear if re-entered mid-record |
| **Stand 080213** | 51–60 yrs | 55 yrs | 82.0 ha | Maturing stand; high Trees % approaching stability; Stable increasingly dominant |

Stand colors match the **Recovery Stage** palette from Section 11:
- Yellow → pioneer/bare
- Orange → shrub/early tree ingrowth
- Green → canopy closed
- Dark blue → maturing/stable

> 📍 **Stand source:** Boundaries and ages are from the [McDonald-Dunn Age Class Distribution feature service](https://services9.arcgis.com/836n6zPvXHaZMNa9/arcgis/rest/services/Age_Class_Distribution_on_the_McDonald_Dunn_Forest/FeatureServer/0), downloaded to `data/macdunn_age_class.geojson`. The 317 stands older than 60 years are excluded from this focused analysis — they are visible in the age raster on the map.


In [ ]:
# ── Representative stand per age class (largest non-agricultural stand by area) ─
# 4 classes covering the early-to-stabilizing recovery window (9-60 yrs)
_STAND_FIDS = {
    450: '9-20 yrs (Stand 020408)',    # Age 15, 81.5 ha — pioneer/early shrub
    401: '21-35 yrs (Stand 010207)',   # Age 21, 75.2 ha — canopy closure
    2:   '36-50 yrs (Stand 010105)',   # Age 49, 53.4 ha — young forest
    224: '51-60 yrs (Stand 080213)',   # Age 55, 82.0 ha — maturing stand
}

harvest_areas = {}
for feat in _stands_gj['features']:
    fid = int(feat['properties']['FID'])
    if fid not in _STAND_FIDS or not feat.get('geometry'):
        continue
    label  = _STAND_FIDS[fid]
    coords = feat['geometry']['coordinates']
    gtype  = feat['geometry']['type']
    if gtype == 'Polygon':
        harvest_areas[label] = ee.Geometry.Polygon(coords)
    else:
        harvest_areas[label] = ee.Geometry.MultiPolygon(coords)

# Re-order by age class (youngest → oldest)
_AGE_CLASS_ORDER = list(_STAND_FIDS.values())
harvest_areas = {k: harvest_areas[k] for k in _AGE_CLASS_ORDER if k in harvest_areas}

print('Representative stand areas (youngest → oldest):')
for name, geom in harvest_areas.items():
    area = geom.area(maxError=100).divide(1e6).getInfo()
    print(f'  {name:40s}: {area:.2f} km2')

# Colors match the Section 11 recovery stage palette
POLY_COLORS = {
    '9-20 yrs (Stand 020408)':   'f7dc6f',  # early seral yellow
    '21-35 yrs (Stand 010207)':  'e67e22',  # mid seral orange
    '36-50 yrs (Stand 010105)':  '27ae60',  # late seral green
    '51-60 yrs (Stand 080213)':  '1a5276',  # maturing dark blue
}

for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': c + '35'},
        name,
    )

Map.addLayer(
    macdunn_boundary_fc,
    {'layerType': 'geeVectorImage', 'strokeColor': 'cccccc', 'strokeWidth': 1,
     'fillColor': '00000000'},
    'McDonald-Dunn Boundary',
    True,
)
Map.setCenter(-123.34, 44.70, 11)
Map.view()


Representative stand areas:
  40-80 yrs (Stand 020503)                : 0.50 km2
  80-120 yrs (Stand 041810)               : 0.43 km2
  200-500 yrs (Stand 080603)              : 0.27 km2
  120-200 yrs (Stand 070414)              : 0.47 km2
  20-40 yrs (Stand 010207)                : 0.38 km2
  9-20 yrs (Stand 020408)                 : 0.41 km2
Adding layer: 40-80 yrs (Stand 020503)
Adding layer: 80-120 yrs (Stand 041810)
Adding layer: 200-500 yrs (Stand 080603)
Adding layer: 120-200 yrs (Stand 070414)
Adding layer: 20-40 yrs (Stand 010207)
Adding layer: 9-20 yrs (Stand 020408)
Adding layer: McDonald-Dunn Boundary
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784663613221


## 6 · Recovery Trajectories — Land Cover Through Time

For each representative stand we compute the **annual percentage of each Land Cover class** from 1985 to 2025. The post-clearcut succession in Oregon Coast Range Douglas-fir forests follows a well-documented pathway:

| Stage | Dominant Cover | Typical Post-Harvest Timing |
|-------|---------------|-----------------------------|
| **Pioneer** | Grass/Forb/Herb | Years 1–3 |
| **Early shrub** | Shrubs (vine maple, red alder, bracken fern) | Years 2–10 |
| **Crown closure** | Trees (Douglas-fir canopy closing in plantations) | Years 8–20 |
| **Closed canopy** | Trees + Tall Trees and Shrubs | Years 18+ |

Because the six stands span an age range of 15–260 years, they represent **different positions in this succession sequence at any given point in time**. The youngest stands (9–20 yrs) should show Grass → Shrub → early Trees within the record. The oldest (200–500 yrs) should show near-constant Trees throughout. Mid-age stands (40–120 yrs) tell the most interesting story: they may show one or more harvest events followed by a recovery arc — or a stable high-Trees signature if the last cut predates 1985.

> 🔍 Each chart is a stacked line plot — the height of each color band shows what percentage of the stand was in that cover class that year. Six charts, one per age class, let you compare how the same forest type looks at different stages of the rotation cycle simultaneously.


In [ ]:
lc_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Land_Cover',
        scale=30,
        area_format='Percentage',
        title=f'Land Cover — {name}',
        chart_type='line',
        stacked=True,
        date_format='YYYY',
        width=950,
        height=440,
    )
    lc_results[name] = result

    if EXPORT_HTML:
        import os as _os
        fname = _os.path.join(_export_dir, f'macdunn_lc_{slug}.html')
        cl.save_chart_html(result['chart'], fname)
        print(f'Saved: {fname}')
    if EXPORT_CSV:
        import os as _os
        csv_fname = _os.path.join(_export_dir, f'macdunn_lc_{slug}.csv')
        result['df'].to_csv(csv_fname)
        print(f'Saved: {csv_fname}')

    result['chart'].show()


### 6a · Inspect the Recovery Data

The `summarize_and_chart()` call returns both a chart and the raw DataFrame. Below we extract the **Trees** column for all four harvest areas and display them in a single table for direct numerical comparison.

In [ ]:
import pandas as pd

# Extract the "Trees" column from each polygon's DataFrame
# Column name may be 'Trees' or include a class-value prefix like '1 — Trees'
trees_pct = {}
for name, result in lc_results.items():
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c),
        None,
    )
    if trees_col:
        trees_pct[name] = df[trees_col].round(1)
    else:
        print(f'Warning: Trees column not found for "{name}". Columns: {df.columns.tolist()}')

if trees_pct:
    trees_df = pd.DataFrame(trees_pct)
    trees_df.index.name = 'Year'
    print('Annual Trees cover (%) by harvest area:\n')
    print(trees_df.to_markdown())


Annual Trees cover (%) by harvest area:

|   Year |   40-80 yrs (Stand 020503) |   80-120 yrs (Stand 041810) |   200-500 yrs (Stand 080603) |   120-200 yrs (Stand 070414) |   20-40 yrs (Stand 010207) |   9-20 yrs (Stand 020408) |
|-------:|---------------------------:|----------------------------:|-----------------------------:|-----------------------------:|---------------------------:|--------------------------:|
|   1985 |                       99.7 |                        99.9 |                        100   |                        100   |                      100   |                     100   |
|   1986 |                       99.9 |                        99.9 |                        100   |                        100   |                      100   |                     100   |
|   1987 |                       99.9 |                        99.9 |                        100   |                        100   |                      100   |                     100   |
|   1988 |    

## 7 · Change Agent Signatures — Tree Removal and Successional Growth

The LCMS `Change` band is the direct record of *what happened* in a pixel each year. The six age-class stands should show distinct change signatures that directly reflect their position in the rotation cycle:

| Stand | Tree Removal in LCMS? | Successional Growth? |
|-------|-----------------------|----------------------|
| **9–20 yrs (Stand 020408, Age 15)** | Yes — spike near end of record (stand was recently harvested) | Emerging — early seral, just transitioning |
| **20–40 yrs (Stand 010207, Age 21)** | Possible — harvest likely falls within or just before the LCMS record | Yes — dominant change class as young stand establishes |
| **40–80 yrs (Stand 020503, Age 74)** | Possible mid-record if a second rotation was completed within the LCMS window | Yes — active recovery signal during early record |
| **80–120 yrs (Stand 041810, Age 103)** | Unlikely unless stand was re-entered during the record | Declining — stand approaching crown closure, Stable increasingly dominant |
| **120–200 yrs (Stand 070414, Age 148)** | Unlikely | Minimal — mature stand; Stable dominant throughout |
| **200–500 yrs (Stand 080603, Age 260)** | Very unlikely | Absent — old-growth remnant; should hold Stable for nearly the entire record |

**What to look for:**

- The **gradient from disturbance-dominated to stability-dominated** as you move from youngest to oldest stands — this directly visualizes the age-class effect on LCMS signals
- **Repeat disturbance** in mid-age stands: a second Tree Removal spike after a period of Successional Growth means a second rotation was completed within the 40-year window
- **The lag** between Tree Removal and the onset of Successional Growth — typically 3–7 years in the Oregon Coast Range


In [ ]:
import os as _os

change_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Change',
        scale=30,
        area_format='Percentage',
        title=f'Change Agents — {name}',
        chart_type='line',
        stacked=False,
        date_format='YYYY',
        width=950,
        height=440,
    )
    change_results[name] = result

    if EXPORT_HTML:
        fname = _os.path.join(_export_dir, f'macdunn_change_{slug}.html')
        cl.save_chart_html(result['chart'], fname)
        print(f'Saved: {fname}')
    if EXPORT_CSV:
        csv_fname = _os.path.join(_export_dir, f'macdunn_change_{slug}.csv')
        result['df'].to_csv(csv_fname)
        print(f'Saved: {csv_fname}')

    result['chart'].show()


## 8 · Combined Comparison — Tree Cover Across Age Classes

Overlaying the **Trees cover percentage** for all six age-class stands on a single chart gives a direct visual test of the chronosequence hypothesis: if LCMS captures the harvest-recovery cycle faithfully, stands should be **rank-ordered by age throughout most of the record**, with older stands consistently showing higher Trees %.

**Expected patterns:**

- **200–500 yrs (Stand 080603)** — near the top throughout; high stable Trees characteristic of a long-uncut stand
- **120–200 yrs (Stand 070414)** — similarly high and stable; well-established mature forest
- **80–120 yrs (Stand 041810)** — high Trees %, possibly a slight dip if the stand was re-entered during the record
- **40–80 yrs (Stand 020503)** — Trees dominant but with a harvest dip visible if a rotation was completed mid-record
- **20–40 yrs (Stand 010207)** — lower Trees % in 1985, rising steadily as the young plantation closes the canopy
- **9–20 yrs (Stand 020408)** — lowest Trees % near the end of the record; still in early seral at the time of the most recent cut

**Rank inversions** — where a younger stand temporarily shows more Trees % than an older one — indicate either unmapped harvests in the older stand, topographic effects on regeneration rate, or non-forest land cover within the stand boundary diluting the tree signal.

> 📈 The spread between the youngest and oldest curves at any given year is a measure of how well LCMS resolves the age-class mosaic of an actively managed forest.


In [ ]:
import plotly.graph_objects as go
import pandas as pd

LINE_COLORS = {
    '9-20 yrs (Stand 020408)':   '#f7dc6f',
    '21-35 yrs (Stand 010207)':  '#e67e22',
    '36-50 yrs (Stand 010105)':  '#27ae60',
    '51-60 yrs (Stand 080213)':  '#1a5276',
}
DASH_STYLES = ['solid', 'dash', 'dot', 'dashdot']

trees_pct = {}
fig = go.Figure()

for (name, result), dash in zip(lc_results.items(), DASH_STYLES):
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c), None
    )
    if trees_col:
        trees_pct[name] = df[trees_col].round(1)
        fig.add_trace(go.Scatter(
            x=df.index,
            y=df[trees_col].values,
            name=name,
            mode='lines+markers',
            line=dict(color=LINE_COLORS[name], width=2.5, dash=dash),
            marker=dict(size=4),
        ))

fig.update_layout(
    title='McDonald-Dunn Research Forest — Tree Cover by Age Class Stand 1985-2025',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='Trees (% of stand area)', range=[0, 100]),
    legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0),
    width=1050, height=550,
    template='plotly_white',
)

if EXPORT_HTML:
    path = _os.path.join(_export_dir, 'macdunn_recovery_comparison.html')
    fig.write_html(path)
    print(f'Saved: {path}')
if EXPORT_CSV and trees_pct:
    path = _os.path.join(_export_dir, 'macdunn_recovery_comparison.csv')
    pd.DataFrame(trees_pct).to_csv(path)
    print(f'Saved: {path}')

fig.show()


## 8b · Tree Canopy Cover (TCC) Trajectories

Annual mean Science TCC (%) for each representative age-class stand, 1985–2025.

In [ ]:
TCC_ASSET = 'projects/gtac-data-publish/assets/TCC/Product_Version/2025-6'
TCC_BAND  = 'Science_Percent_Tree_Canopy_Cover'

tcc = (
    ee.ImageCollection(TCC_ASSET)
    .filter(ee.Filter.inList('study_area', ['CONUS']))
    .filterBounds(study_area)
)

tcc_results = {}

for name, geom in harvest_areas.items():
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    result = cl.summarize_and_chart(
        tcc.filterBounds(geom),
        geometry=geom,
        band_names=TCC_BAND,
        scale=30,
        area_format='Mean',
        title=f'Tree Canopy Cover (TCC) — {name}',
        chart_type='line',
        date_format='YYYY',
        width=950,
        height=440,
    )
    tcc_results[name] = result

    if EXPORT_HTML:
        fname = _os.path.join(_export_dir, f'macdunn_tcc_{slug}.html')
        cl.save_chart_html(result['chart'], fname)
        print(f'Saved: {fname}')
    if EXPORT_CSV:
        csv_fname = _os.path.join(_export_dir, f'macdunn_tcc_{slug}.csv')
        result['df'].to_csv(csv_fname)
        print(f'Saved: {csv_fname}')

    result['chart'].show()

# ── Combined TCC comparison ───────────────────────────────────────────────────
fig_tcc = go.Figure()

for (name, result), dash in zip(tcc_results.items(), DASH_STYLES):
    df = result['df']
    tcc_col = next(
        (c for c in df.columns if 'Canopy' in c or 'canopy' in c),
        df.columns[0] if len(df.columns) else None,
    )
    if tcc_col:
        fig_tcc.add_trace(go.Scatter(
            x=df.index, y=df[tcc_col].values,
            name=name, mode='lines+markers',
            line=dict(color=LINE_COLORS[name], width=2.5, dash=dash),
            marker=dict(size=4),
        ))

fig_tcc.update_layout(
    title='McDonald-Dunn Research Forest — Science TCC by Age Class Stand 1985-2025',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='Mean Tree Canopy Cover (%)', range=[0, 100]),
    legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0),
    width=1050, height=550,
    template='plotly_white',
)

if EXPORT_HTML:
    path = _os.path.join(_export_dir, 'macdunn_tcc_comparison.html')
    fig_tcc.write_html(path)
    print(f'Saved: {path}')
if EXPORT_CSV:
    tcc_rows = {}
    for name, result in tcc_results.items():
        tcc_col = next((c for c in result['df'].columns if 'Canopy' in c or 'canopy' in c),
                       result['df'].columns[0] if len(result['df'].columns) else None)
        if tcc_col:
            tcc_rows[name] = result['df'][tcc_col]
    if tcc_rows:
        path = _os.path.join(_export_dir, 'macdunn_tcc_comparison.csv')
        pd.DataFrame(tcc_rows).to_csv(path)
        print(f'Saved: {path}')

fig_tcc.show()


## 8c · TCC + Change Agent — Side-by-Side per Stand

Two-panel subplot per age-class stand: Science TCC % on top, Tree Removal and Successional Growth % below. Shared x-axis lets you read the harvest signal and canopy response together.

In [ ]:
from plotly.subplots import make_subplots

for name, geom in harvest_areas.items():
    if name not in change_results or name not in tcc_results:
        print(f'Skipping {name} — missing data')
        continue

    change_df = change_results[name]['df']
    tcc_df    = tcc_results[name]['df']

    tcc_col     = next((c for c in tcc_df.columns    if 'Canopy' in c or 'canopy' in c),    tcc_df.columns[0]    if len(tcc_df.columns)    else None)
    removal_col = next((c for c in change_df.columns if 'Removal' in c or 'removal' in c),  None)
    growth_col  = next((c for c in change_df.columns if 'Successional' in c or 'Growth' in c), None)

    fig_combo = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=('Science Tree Canopy Cover (%)', 'LCMS Change Agents (% of area)'),
        vertical_spacing=0.14,
        row_heights=[0.45, 0.55],
    )

    if tcc_col:
        fig_combo.add_trace(go.Scatter(
            x=tcc_df.index, y=tcc_df[tcc_col].values,
            name='Science TCC %', mode='lines+markers',
            line=dict(color='#27ae60', width=2.5), marker=dict(size=4),
        ), row=1, col=1)

    for col, label, color in [
        (removal_col, 'Tree Removal',        '#e84c1e'),
        (growth_col,  'Successional Growth', '#4caf50'),
    ]:
        if col:
            fig_combo.add_trace(go.Scatter(
                x=change_df.index, y=change_df[col].values,
                name=label, mode='lines+markers',
                line=dict(color=color, width=2), marker=dict(size=3),
            ), row=2, col=1)

    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    fig_combo.update_layout(
        title=f'TCC and Change Agents — {name}',
        yaxis=dict(title='TCC (%)', range=[0, 100]),
        yaxis2=dict(title='% of area', range=[0, 100]),
        xaxis2=dict(title='Year', tickmode='linear', dtick=5),
        height=680, width=1000,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='top', y=-0.10, xanchor='left', x=0),
    )

    if EXPORT_HTML:
        fname = _os.path.join(_export_dir, f'macdunn_tcc_change_{slug}.html')
        fig_combo.write_html(fname)
        print(f'Saved: {fname}')

    fig_combo.show()


## 9 · Land Use Transitions — Sankey Diagram

The Land Use Sankey shows how land area has shifted between use classes across the whole study area at key time steps. For McDonald-Dunn — unlike a protected or wilderness site — the Forest class should remain the dominant category throughout, with relatively small flows that reflect active management decisions rather than broad land-use conversion.

The transition years below align with management and policy inflection points:
- **1985** → LCMS baseline; active commercial harvest throughout both blocks
- **1995** → post-1994 Northwest Forest Plan — **note that McDonald-Dunn is OSU land, not USFS, so it was unaffected**; contrast with what a comparable Siuslaw NF analysis would show
- **2010** → updated OSU management plan era; diverse age classes across the forest
- **2025** → present; ongoing active management with recent harvests visible in the LCMS record

> 💡 **The interesting contrast**: Adjacent USFS lands (Siuslaw, Willamette NFs) would show large flows *out of Forest land use* after the 1994 Northwest Forest Plan. McDonald-Dunn, as a university forest, continued active management — the Sankey may show a stable Forest block throughout, making the policy boundary literally visible on a map.

In [ ]:
lu_sankey = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Land_Use',
    scale=120,
    area_format='Percentage',
    title=f'McDonald-Dunn Research Forest — Land Use Transitions 1985 -> 1995 -> 2010 -> {END_YEAR}',
    sankey=True,
    transition_periods=[1985, 1995, 2010, END_YEAR],
    min_percentage=0.5,
    width=1000,
    height=600,
)

if EXPORT_HTML:
    path = _os.path.join(_export_dir, 'macdunn_land_use_sankey.html')
    cl.save_chart_html(lu_sankey['chart'], path)
    print(f'Saved: {path}')
if EXPORT_CSV and 'df' in lu_sankey:
    path = _os.path.join(_export_dir, 'macdunn_land_use_sankey.csv')
    lu_sankey['df'].to_csv(path)
    print(f'Saved: {path}')

display(HTML(lu_sankey['chart']))


### 9a · Land Use Transition Matrices

The raw transition numbers behind the Sankey — useful for quantifying exactly how much area moved between classes in each period.

In [ ]:
if 'matrix' in lu_sankey:
    for period_key, mat in lu_sankey['matrix'].items():
        print(f'### {period_key}')
        print(mat.to_markdown())
        print()


### 1985 → 1995
|                          |   Agriculture |   Developed |   Forest |   Other |   Rangeland or Pasture |   Non-Processing Area Mask |
|:-------------------------|--------------:|------------:|---------:|--------:|-----------------------:|---------------------------:|
| Agriculture              |         32.09 |        0.11 |     2.01 |    0    |                   0.15 |                          0 |
| Developed                |          0    |        1.62 |     0.02 |    0    |                   0.02 |                          0 |
| Forest                   |          0.82 |        0.03 |    61.89 |    0.01 |                   0.12 |                          0 |
| Other                    |          0    |        0    |     0    |    0.04 |                   0    |                          0 |
| Rangeland or Pasture     |          0.04 |        0.02 |     0.14 |    0    |                   0.83 |                          0 |
| Non-Processing Area Mask |          0    |  

## 10 · Interactive Time-Lapse — 40 Years of Change and Recovery

The time-lapse steps through each year's LCMS Change and Land Cover bands, letting you see *where* Tree Removal and recovery occurred spatially across the McDonald-Dunn landscape — not just *how much* area they affected. The management zone outlines are overlaid as reference.

**What to look for:**
- **Scattered Tree Removal events** appearing in *different* compartments across different years — the spatial mosaic of a rotation management system, fundamentally different from a single large wildfire or clearcut
- The **Shrub → Trees** transition progressing through each harvest unit at different times, reflecting the age-class mosaic built up by decades of rotation harvest
- **Repeat disturbance** in older polygons: if Tree Removal appears twice in the same location decades apart, that is the rotation harvest cycle completing a second pass
- Adjacent unharvested old-growth remnants or long-uncut patches that hold "Stable" throughout, providing visual contrast to the actively managed blocks

> ⏱️ This may take 30–60 seconds to load — it is rendering 41 annual layers.

In [ ]:
Map.clearMap()

Map.addTimeLapse(
    lcms.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    'Change Agent (Annual)',
    visible=True,
)
Map.addTimeLapse(
    lcms.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover (Annual)',
    visible=False,
)

for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name, False,
    )

Map.addLayer(
    macdunn_boundary_fc,
    {'layerType': 'geeVectorImage', 'strokeColor': 'cccccc', 'strokeWidth': 1,
     'fillColor': '00000000'},
    'McDonald-Dunn Boundary',
)
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'McDonald-Dunn Study Area',
)
Map.setCenter(-123.34, 44.70, 11)
Map.view()


Adding layer: Change Agent (Annual)
Adding layer: Land Cover (Annual)
Adding layer: 40-80 yrs (Stand 020503)
Adding layer: 80-120 yrs (Stand 041810)
Adding layer: 200-500 yrs (Stand 080603)
Adding layer: 120-200 yrs (Stand 070414)
Adding layer: 20-40 yrs (Stand 010207)
Adding layer: 9-20 yrs (Stand 020408)
Adding layer: McDonald-Dunn Boundary
Adding layer: McDonald-Dunn Study Area
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784664165838


## 11 · Bonus — Classify Recovery Stages Across the Study Area

Rather than looking at individual polygons, here we classify *every pixel* in the study area by how many years (out of 41) it was mapped as Trees. This "years-as-trees" metric is a proxy for successional maturity — but in an actively managed forest it also maps directly onto the rotation age-class structure: a pixel that scores 40+ has been in continuous tree cover (old plantation or old-growth remnant); a pixel that scores 5 was recently clearcut or is perpetually in a non-tree cover class.

| Stage | Years as Trees | Interpretation for McDonald-Dunn |
|-------|---------------|----------------------------------|
| **Early seral** | 0–5 | Recently clearcut or non-forested; grass or shrub dominant |
| **Mid seral** | 6–15 | Young plantation establishing; canopy not yet closed |
| **Late seral** | 16–29 | Dense young forest; typical rotation harvest age class |
| **Mature/established stand** | 30–41 | Closed-canopy throughout nearly the entire record; 1950s–1960s plantations or old-growth remnants |

In an actively managed forest the spatial pattern of these stages should map closely onto the documented harvest rotation mosaic — each management compartment should cluster in a recovery stage reflecting when it was last cut. This makes "years-as-trees" a simple but powerful visual summary of the management age-class structure across the entire forest.

In [ ]:
Map.clearMap()

years_as_trees = (
    lcms.select('Land_Cover')
    .map(lambda img: img.eq(1).rename('is_trees'))
    .sum()
    .rename('years_as_trees')
)

recovery_stage = (
    ee.Image(1)
    .where(years_as_trees.gt(5),  2)
    .where(years_as_trees.gt(15), 3)
    .where(years_as_trees.gt(29), 4)
    .rename('recovery_stage')
    .updateMask(years_as_trees.gte(0))
)
recovery_stage = recovery_stage.set({
    'recovery_stage_class_values':  [1, 2, 3, 4],
    'recovery_stage_class_names':   [
        'Early seral (0-5 yrs Trees)',
        'Mid seral (6-15 yrs Trees)',
        'Late seral (16-29 yrs Trees)',
        'Mature/established stand (30-41 yrs Trees)',
    ],
    'recovery_stage_class_palette': ['f7dc6f', 'e67e22', '27ae60', '1a5276'],
})

Map.addLayer(
    lcms_most_recent.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}', False,
)
Map.addLayer(
    recovery_stage,
    {'autoViz': True, 'canAreaChart': True},
    'Recovery Stage (years as Trees, 1985-2025)',
    True,
)

for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name, True,
    )

Map.addLayer(
    macdunn_boundary_fc,
    {'layerType': 'geeVectorImage', 'strokeColor': 'cccccc', 'strokeWidth': 1,
     'fillColor': '00000000'},
    'McDonald-Dunn Boundary',
)
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'McDonald-Dunn Study Area',
)
Map.setCenter(-123.34, 44.70, 11)
Map.view()


Adding layer: Land Cover 2024
Adding layer: Recovery Stage (years as Trees, 1985-2025)
Adding layer: 40-80 yrs (Stand 020503)
Adding layer: 80-120 yrs (Stand 041810)
Adding layer: 200-500 yrs (Stand 080603)
Adding layer: 120-200 yrs (Stand 070414)
Adding layer: 20-40 yrs (Stand 010207)
Adding layer: 9-20 yrs (Stand 020408)
Adding layer: McDonald-Dunn Boundary
Adding layer: McDonald-Dunn Study Area
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784664165879


## 12 · Key Takeaways and Next Steps

### What LCMS tells us about an actively managed Oregon Coast Range research forest

After running this notebook you should be able to answer:

1. **Harvest detection** — Does LCMS reliably detect the Tree Removal events in the McDonald-Dunn management zones? Which silvicultural treatments (clearcut vs. shelterwood vs. selection) produce the clearest LCMS signal? Do smaller selection harvests appear as reduced Tree Removal intensity rather than a distinct spike?

2. **Recovery rate** — How many years does it take for the Trees land-cover class to re-dominate each polygon after harvest? Is the Coast Range faster than comparable western Cascades analyses, as expected given the lower-elevation climate?

3. **Rotation visibility** — Can you identify a second Tree Removal event in any of the older polygons — evidence that a second rotation was completed within the 1985–2025 LCMS window? This is the defining feature of a managed forest that would not appear in a protected LTER analysis.

4. **Chronosequence integrity** — Do the four management-zone polygons form a coherent chronosequence on the combined Trees-recovery chart (oldest at top, youngest at bottom throughout the record)? Any violations may reflect unmapped re-harvests or strong topographic/aspect effects on recovery rate.

5. **Land use stability** — Does the Sankey show the Forest land-use class holding stable throughout, consistent with continuous OSU forestry management? How does this contrast with what you would expect from an adjacent USFS forest subject to the 1994 Northwest Forest Plan?

6. **Recovery stage map** — Does the "years-as-trees" spatial pattern correspond to the known management rotation mosaic? High correspondence supports using LCMS as a forest age-class mapping tool for non-federal managed forests across the Pacific Northwest.

---

### Going further

| Idea | How |
|------|-----|
| Use official harvest unit polygons | Request GIS data from the OSU Research Forest Business Office at Peavy Arboretum, or see the [2025 Management Plan](https://cf.forestry.oregonstate.edu/our-forests/2025-mcdonald-dunn-forest-plan) |
| Compare with adjacent USFS land | Replace `study_area` with `sal.getUSFSForests(forest_name='Siuslaw')` to see the 1994 Northwest Forest Plan management shift in contrast to OSU's continuous management |
| Add topographic controls | Load `USGS/SRTMGL1_003`, compute slope and aspect; compare Trees% recovery rate by topographic position across Coast Range terrain |
| Extend to drier eastern Oregon | Duplicate the notebook with a Malheur or Ochoco NF bounding box — drier climate, fire-driven disturbance, much slower recovery |
| Overlay MTBS fire perimeters | Load MTBS from the GEE catalog — while the Coast Range is wet, the 2020 Labor Day fires burned surprisingly close; check whether any perimeter intersects the study area |
| Validate harvest detection | If you have official harvest record shapefiles, overlay them on the "Ever Harvested" layer to compute LCMS detection rates by silvicultural treatment type |
| Export annual land cover maps | Use the EE batch export API (`ee.batch.Export.image.toDrive(...)`) for GeoTIFF outputs suitable for GIS integration with the OSU forest management records |

---

### Data citation

> USFS GTAC. (2025). *Landscape Change Monitoring System v2025-11*. USDA Forest Service, Geospatial Technology and Applications Center. [https://www.fs.usda.gov/lcms](https://www.fs.usda.gov/lcms)

> Google Earth Engine catalog: `projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11`

> McDonald-Dunn Research Forest, Oregon State University College of Forestry. [https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests)